# SVR (Support Vector Regression)

SVR es la variante de **regresión** de SVM. En vez de separar clases, ajusta una función que mantiene los errores dentro de un margen de tolerancia, penalizando solo los puntos que se salen. Sigue usando el **kernel trick** para relaciones no lineales.

Regresión sobre `Hitters` (predecir `log(Salary)`). Probamos distintos **kernels** y valores de **C** para ver cuánto impactan.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

## Datos

`Hitters` sin nulos, solo variables numéricas. Target = `log(Salary)`. **Estandarizamos** (SVR trabaja con distancias).

In [ ]:
data_complete = pd.read_csv('../datasets/Hitters.csv').dropna()

data_columns = ['AtBat','Hits','HmRun','Runs','RBI','Walks','Years','CAtBat','CHits','CHmRun','CRuns','CRBI','CWalks',
                'PutOuts','Assists','Errors','Salary']
data = data_complete.loc[:, data_columns]

X = data.drop('Salary', axis=1)
y = np.log(data['Salary'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=127)

In [ ]:
scaler = StandardScaler()
X_train_scl = scaler.fit_transform(X_train)
X_test_scl = scaler.transform(X_test)

## Modelo base: SVR por defecto

`SVR()` usa kernel **RBF** y `C=1` por defecto. Reportamos R2, MSE y RMSE.

In [ ]:
model = SVR()
model.fit(X_train_scl, y_train)
preds = model.predict(X_test_scl)

reg_score = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(reg_score)
print(mse)
print(rmse)

## Variando hiperparámetros

`C` regula la penalización de los errores, y `kernel` cambia la forma de la función. Comparamos contra el modelo base.

In [ ]:
# C mas chico (mas regularizacion)
model = SVR(C=0.1)
model.fit(X_train_scl, y_train)
preds = model.predict(X_test_scl)

reg_score = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(reg_score)
print(mse)
print(rmse)

In [ ]:
# kernel sigmoide
model = SVR(kernel='sigmoid')
model.fit(X_train_scl, y_train)
preds = model.predict(X_test_scl)

reg_score = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(reg_score)
print(mse)
print(rmse)

In [ ]:
# kernel polinomico
model = SVR(kernel='poly')
model.fit(X_train_scl, y_train)
preds = model.predict(X_test_scl)

reg_score = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(reg_score)
print(mse)
print(rmse)

## Comparación

R2 sobre test (mayor es mejor):

| Configuración | R2 |
|---------------|-----|
| `SVR()` (RBF, C=1) | ~0.65 |
| `SVR(C=0.1)` | ~0.58 |
| `SVR(kernel='sigmoid')` | ~-12.8 |
| `SVR(kernel='poly')` | ~0.39 |

El **kernel y C cambian drásticamente** el resultado: el RBF por defecto es el mejor, bajar `C` regulariza de más, el polinómico se queda corto y el **sigmoide es catastrófico** (R2 muy negativo, peor que predecir la media). Esto motiva **buscar los hiperparámetros** de forma sistemática (Grid/Random/Bayesian Search) en vez de a mano.